In [0]:
%run /Workspace/Users/riley.day@kwa-analytics.com/Databricks-Certified-Data-Engineer-Associate/Includes/Copy-Datasets

In [0]:
select * from customers

In [0]:
DESCRIBE CUSTOMERS

In [0]:
select customer_id, profile:first_name, profile:address:country from customers

In [0]:
select from_json(profile) as profile_struct from customers

In [0]:
select profile from customers limit 1

In [0]:
create or replace temp view parsed_customers as 
select customer_id, from_json(profile, schema_of_json('{"first_name":"Susana","last_name":"Gonnely","gender":"Female","address":{"street":"760 Express Court","city":"Obrenovac","country":"Serbia"}}')) as profile_struct from customers;

select * from parsed_customers

In [0]:
DESCRIBE parsed_customers

In [0]:
select customer_id, profile_struct.first_name, profile_struct.address.country from parsed_customers

In [0]:
CREATE OR REPLACE TEMP VIEW customers_final as 
select customer_id, profile_struct.* from 
parsed_customers;

select * from customers_final

In [0]:
select order_id, customer_id, books from orders

In [0]:
select order_id, customer_id, explode(books) as book from orders

In [0]:
select customer_id, collect_set(order_id) as orders_set,
collect_set(books.book_id) as books_set
from orders
group by customer_id

In [0]:
select customer_id, collect_set(books.book_id) as before_flatten, array_distinct(flatten(collect_set(books.book_id))) as after_flatten
from orders
group by customer_id

In [0]:
CREATE OR REPLACE VIEW orders_enriched AS
SELECT *
FROM (
    SELECT *, explode(books) as book from orders
) o 
INNER JOIN books b
ON o.book.book_id=b.book_id;

select * from orders_enriched

In [0]:
CREATE OR REPLACE TEMP VIEW orders_updates
AS select * from parquet.`${dataset.bookstore}/orders-new`;

select * from orders
union
select * from orders_updates

In [0]:
select * from orders
intersect
select * from orders_updates

In [0]:
select * from orders
minus
select * from orders_updates

In [0]:
CREATE OR REPLACE TABLE transactions AS
SELECT * FROM (
    SELECT customer_id,
    book.book_id as book_id,
    book.quantity as quantity
    from orders_enriched)
    PIVOT (
    sum(quantity) for book_id in (
        'B01','B02','B03','B04','B05','B06','B07','B08','B09','B10','B11','B12'
    )
)

In [0]:
select * from transactions